# Imports

In [1]:

import pandas as pd
import logging
from datetime import datetime
import os
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient


# READER

In [2]:
# Lees de parquet file in als een pandas dataframe
df = pd.read_parquet('dataset/yellow_tripdata_2025-01.parquet')

# Maak de logs folder aan als die nog niet bestaat
os.makedirs('./logs', exist_ok=True)

# Schrijf info over de data naar een log file
with open('./logs/data_reader.log', 'a') as f:
    f.write(f"Rijen: {df.shape[0]}, kolommen: {df.shape[1]}\n")
    # Hoeveel lege waarden per kolom
    f.write(f"Ontbrekende waardes:\n{df.isnull().sum()[df.isnull().sum() > 0]}\n")
        # Hoeveel dubbele rijen
    f.write(f"Dubbelen waardes: {df.duplicated().sum()}\n")
        # Statistieken zoals gemiddelde, min, max per kolom
    f.write(f"Stats:\n{df.describe()}\n")

    # Sla de data op als tijdelijk bestand zodat de validator het kan inlezen
df.to_parquet('/tmp/taxi_raw.parquet', index=False)


# VALIDATOR 

In [3]:
# Lees de ruwe data in die de reader heeft opgeslagen
df = pd.read_parquet('/tmp/taxi_raw.parquet')


# Hulpfunctie om eenvoudig naar de log te schrijven
def log(msg):
    with open('./logs/data_validation.log', 'a', encoding='UTF-8') as f:
        f.write(msg + '\n')


# Schrijf een scheiding en tijdstip in de log
log('=' * 60)
log(f"Validatie run — {pd.Timestamp.now()}")
log('=' * 60)


# List of all mandatory columns
mandatory = [
    'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count',
    'trip_distance', 'PULocationID', 'DOLocationID',
    'payment_type', 'fare_amount', 'total_amount'
]


# Loop over elke verplichte kolom en controleer op ontbrekende waarden
for col in mandatory:
    nulls = df[col].isna()
    if nulls.any():
        log(
            f"{col}: {nulls.sum()} missing values at rows {df.index[nulls].tolist()}")


# Controleer passenger_count: moet tussen 1 en 8 liggen
invalid = df[df['passenger_count'].notna() & (
    (df['passenger_count'] < 1) | (df['passenger_count'] > 8))]
if not invalid.empty:
    log(f"passenger_count: {len(invalid)} out-of-range values (1-8)")


# Controleer trip_distance: mag niet negatief zijn
invalid = df[df['trip_distance'] < 0]
if not invalid.empty:
    log(f"trip_distance: {len(invalid)} negative values")


# Controleer payment_type: moet een waarde zijn uit {0,1,2,3,4,5,6}
invalid = df[~df['payment_type'].isin([0, 1, 2, 3, 4, 5, 6])]
if not invalid.empty:
    log(f"payment_type: {len(invalid)} invalid values")


# Controleer fare_amount en total_amount: mogen niet negatief zijn
for col in ['fare_amount', 'total_amount']:
    invalid = df[df[col] < 0]
    if not invalid.empty:
        log(f"{col}: {len(invalid)} negative values")


# Controleer datetime-volgorde: dropoff moet na pickup zijn
pickup = pd.to_datetime(df['tpep_pickup_datetime'], errors='coerce')
dropoff = pd.to_datetime(df['tpep_dropoff_datetime'], errors='coerce')
invalid = df[(dropoff <= pickup) & pickup.notna() & dropoff.notna()]
if not invalid.empty:
    log(f"datetime order: {len(invalid)} rows where dropoff <= pickup")


# Verwijder rijen waar verplichte kolommen leeg zijn
df = df.dropna(subset=mandatory)


# Behoud alleen rijen waar passenger_count tussen 1 en 8 ligt
df = df[df['passenger_count'].between(1, 8)]


# Behoud alleen rijen waar trip_distance groter is dan 0
df = df[df['trip_distance'] > 0]


# Behoud alleen rijen waar payment_type geldig is
df = df[df['payment_type'].isin([0, 1, 2, 3, 4, 5, 6])]


# Behoud alleen rijen waar fare_amount niet negatief is
df = df[df['fare_amount'] >= 0]


# Behoud alleen rijen waar total_amount niet negatief is
df = df[df['total_amount'] >= 0]


# Verwijder rijen waar dropoff vóór of gelijk is aan pickup
pickup = pd.to_datetime(df['tpep_pickup_datetime'], errors='coerce')
dropoff = pd.to_datetime(df['tpep_dropoff_datetime'], errors='coerce')
df = df[~((dropoff <= pickup) & pickup.notna() & dropoff.notna())]


# Schrijf afronding van validatie naar de log
log("Validatie compleet.\n")


# Sla de opgeschoonde data op zodat de processor verder kan werken
df.to_parquet('/tmp/taxi_validated.parquet', index=False)

# Processor 


In [4]:
# Lees de gevalideerde data in van de validator
df = pd.read_parquet('/tmp/taxi_validated.parquet')

# Verwijder kolommen die we niet nodig hebben
df = df.drop(columns=['VendorID', 'store_and_fwd_flag', 'RatecodeID'])

# Zet de datumkolommen om naar datetime zodat we ermee kunnen rekenen
pickup = pd.to_datetime(df['tpep_pickup_datetime'])
dropoff = pd.to_datetime(df['tpep_dropoff_datetime'])

# Bereken hoe lang de rit duurde in minuten
# dropoff - pickup geeft een tijdsverschil
# .dt.total_seconds() / 60 zet dat om naar minuten
df['trip_duration_minutes'] = (dropoff - pickup).dt.total_seconds() / 60

# Haal het jaar en de maand uit de pickup-datum
df['pickup_year'] = pickup.dt.year
df['pickup_month'] = pickup.dt.month

# Bereken de gemiddelde snelheid in mijl per uur
# afstand / tijd (in uren) = snelheid
# .where() → alleen berekenen als duur > 0, anders deling door 0
df['average_speed_mph'] = df['trip_distance'] / \
    (df['trip_duration_minutes'] / 60).where(df['trip_duration_minutes'] > 0)

# Bereken hoeveel opbrengst per mijl werd verdiend
# .where() → alleen berekenen als afstand > 0, anders deling door 0
df['revenue_per_mile'] = df['total_amount'] / \
    df['trip_distance'].where(df['trip_distance'] > 0)

# Maak een categorie op basis van de afstand
# Short  = minder dan 2 mijl
# Medium = tussen 2 en 10 mijl
# Long   = meer dan 10 mijl
df['trip_distance_category'] = pd.cut(
    df['trip_distance'],
    bins=[-float('inf'), 2, 10, float('inf')],
    labels=['Short', 'Medium', 'Long']
)

# Maak een categorie op basis van de ritprijs
# Low    = minder dan $20
# Medium = tussen $20 en $50
# High   = meer dan $50
df['fare_category'] = pd.cut(
    df['fare_amount'],
    bins=[-float('inf'), 20, 50, float('inf')],
    labels=['Low', 'Medium', 'High']
)

# Maak een categorie op basis van het uur van de dag
# Night     = 0 tot 5 uur
# Morning   = 6 tot 11 uur
# Afternoon = 12 tot 17 uur
# Evening   = 18 tot 23 uur
df['trip_time_of_day'] = pd.cut(
    pickup.dt.hour,
    bins=[-1, 5, 11, 17, 23],
    labels=['Night', 'Morning', 'Afternoon', 'Evening']
)

# Sla de verwerkte data op zodat de backup-validator en writer die kunnen inlezen
df.to_parquet('/tmp/taxi_processed.parquet', index=False)

# WRITER

In [ ]:
# Laad de omgevingsvariabelen uit het .env-bestand in de huidige omgeving
load_dotenv()

# Sla het DataFrame op als Parquet-bestand lokaal zonder de pandas-index mee te schrijven
df.to_parquet('output/yellow_tripdata_2025-01.parquet', index=False)

# Maak een verbinding met Azure Blob Storage via de connection string uit de omgevingsvariabelen
client = BlobServiceClient.from_connection_string(os.getenv("AZURE_STORAGE_CONNECTION_STRING"))
# Haal een referentie op naar het specifieke blob-bestand binnen de 'yellow-taxi-data' container
blob = client.get_blob_client(container="yellow-taxi-data", blob="yellow_tripdata_2025-01.parquet")

# Open het lokaal opgeslagen Parquet-bestand in binaire leesmodus
with open("output/yellow_tripdata_2025-01.parquet", "rb") as f:
    # Upload de bestandsinhoud naar Azure Blob Storage en overschrijf een eventueel bestaande blob
    blob.upload_blob(f, overwrite=True)

# Bevestig in de console dat het bestand zowel lokaal als op Azure is opgeslagen
print("Done. Written locally and to Azure.")